## School Info 
* number of schools per suburb

In [ ]:
import pandas as pd 
 
# Extracting school info:
school_info = pd.read_csv("data/dv402-SchoolLocations2025.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'data/dv402-SchoolLocations2025.csv'

In [2]:
# Standardising column names to snake_case:
school_info.columns = school_info.columns.str.strip().str.lower().str.replace(" ", "_")

# Check for repeated columns:
num_dup_school_info = school_info.duplicated().sum()
print(f"There are {num_dup_school_info} duplicate columns.")

# See if there are any missing values:
num_missing = school_info.isna().sum()
print(f"The missing values are:\n-----------------------------\n{num_missing}")

# Drop rows with missing coordinates (X/Y)
school_info = school_info.dropna(subset=['x', 'y'])

# Standardise address_town column
school_info['address_town'] = school_info['address_town'].str.strip().str.upper()

# Standardise school_type (ie. primary/secondar/etc.)
school_info['school_type'] = school_info['school_type'].str.strip().str.lower()


There are 0 duplicate columns.
The missing values are:
-----------------------------
education_sector            0
entity_type                 0
school_no                   0
school_name                 0
school_type                 0
school_status               0
address_line_1              0
address_line_2           2289
address_town                0
address_state               0
address_postcode            0
postal_address_line_1       0
postal_address_line_2    2285
postal_town                 0
postal_state                0
postal_postcode             0
full_phone_no               1
region                      0
area                        0
lga_id                      0
lga_name                    0
lga_type                    0
x                           0
y                           0
dtype: int64


In [ ]:
# Calculate the total number of schools per suburb
school_counts = school_info.groupby('address_town').size().reset_index(name='num_schools')

# Calculate the number of primary and secondary schools per suburb
primary_counts = school_info[school_info['school_type'].isin(['primary', 'pri/sec'])].groupby('address_town').size().reset_index(name='num_primary')
secondary_counts = school_info[school_info['school_type'].isin(['secondary', 'pri/sec'])].groupby('address_town').size().reset_index(name='num_secondary')

# Combine these counts, unifying the table by suburb (i.e. address_town) 
suburb_features = school_counts.merge(primary_counts, on='address_town', how='left')
suburb_features = suburb_features.merge(secondary_counts, on='address_town', how='left')
suburb_features.fillna(0, inplace=True)  


In [4]:
# See how many primary/secondary schools are in each suburb
suburb_features

,address_town,num_schools,num_primary,num_secondary
0,ABBOTSFORD,2,2.0,1.0
1,ABERFELDIE,3,2.0,1.0
2,AINTREE,3,1.0,1.0
3,AIREYS INLET,1,1.0,0.0
4,AIRLY,1,1.0,0.0
...,...,...,...,...
937,YEA,3,2.0,1.0
938,YERING,1,1.0,0.0
939,YINNAR,1,1.0,0.0
940,YINNAR SOUTH,1,1.0,0.0


In [ ]:
# Save this suburb info to a new csv within data
suburb_features.to_csv("data/suburb_features_by_school.csv", index=False)


OSError: Cannot save file into a non-existent directory: '..\data'

## PTV Info

In [ ]:
# Extracting Metropolitan Train (2) 'STOP' data:
temp_path = "data/metro_train_stops.txt"
metro_train_stops = pd.read_csv(temp_path)

In [ ]:
metro_train_stops  

,stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,level_id,platform_code
0,10117,Jordanville Station,-37.873763,145.112473,NaN,vic:rail:JOR,1.0,Level 0,1
1,10920,Flagstaff Station,-37.811880,144.956043,NaN,vic:rail:FGS,1.0,Level -2,1
2,10921,Flagstaff Station,-37.811725,144.955968,NaN,vic:rail:FGS,1.0,Level -2,2
3,10922,Melbourne Central Station,-37.809973,144.962513,NaN,vic:rail:MCE,1.0,Level -2,1
4,10923,Melbourne Central Station,-37.809865,144.962505,NaN,vic:rail:MCE,1.0,Level -2,2
...,...,...,...,...,...,...,...,...,...
1053,vic:rail:WTT_EN1,Service Rd,-37.663997,145.181773,2.0,vic:rail:WTT,1.0,Level 0,NaN
1054,vic:rail:WTT_PR1,Park & Ride 1,-37.664410,145.181649,3.0,vic:rail:WTT,NaN,Level 0,NaN
1055,vic:rail:WTT_PR2,Park & Ride 2,-37.664985,145.181499,3.0,vic:rail:WTT,NaN,Level 0,NaN
1056,vic:rail:YMN,Yarraman Railway Station,-37.978255,145.191600,1.0,NaN,1.0,NaN,NaN


## Postcode data

In [ ]:
postcode_path = "data/au_postcodes.csv"

postcodes = pd.read_csv(postcode_path)

## Euclidean distance (using latitude & longitude) between public transport and different places within each Victorian postcode

### Metro Trains

In [32]:
from scipy.spatial import cKDTree
import numpy as np

# Build a tree of metro train stop latitude/longitudes
stop_coords = np.vstack([metro_train_stops["stop_lat"], metro_train_stops["stop_lon"]]).T
tree = cKDTree(stop_coords)

# Find the nearest metro train stop to each place_name in postcodes
postcode_coords = np.vstack([postcodes["latitude"], postcodes["longitude"]]).T
distances, indices = tree.query(postcode_coords, k=1)  

# Save results back to postcodes dataframe
postcodes["nearest_metro_train_stop_distance"] = distances
postcodes["nearest_metro_train_stop_index"] = indices  

# Having a look at the updated postcodes df
print(postcodes.head())

   postcode      place_name state_name state_code  latitude  longitude  \
0      3000       Melbourne   Victoria        VIC  -37.8140   144.9633   
1      3001       Melbourne   Victoria        VIC  -37.8140   144.9633   
2      3002  East Melbourne   Victoria        VIC  -37.8167   144.9879   
3      3003  West Melbourne   Victoria        VIC  -37.8101   144.9500   
4      3004       Melbourne   Victoria        VIC  -37.8140   144.9633   

   accuracy  nearest_metro_stop_distance  nearest_metro_stop_index  \
0         4                     0.003649                       828   
1         4                     0.003649                       828   
2         4                     0.002773                       774   
3         4                     0.005969                       722   
4         4                     0.003649                       828   

   nearest_metro_stop_lat  nearest_metro_stop_lon  \
0              -37.810535              144.962157   
1              -37.810535   

### Metro Trams

In [ ]:
# Extracting Metropolitan Tram (3) 'STOP' data:
temp_path = "data/metro_tram_stops.txt"
metro_tram_stops = pd.read_csv(temp_path)

<>:2: SyntaxWarning: invalid escape sequence '\k'
<>:2: SyntaxWarning: invalid escape sequence '\k'
C:\Users\kaira\AppData\Local\Temp\ipykernel_25080\1372823575.py:2: SyntaxWarning: invalid escape sequence '\k'
  temp_path = "C:\\Users\kaira\\OneDrive\\Documents\\UniMelb (Main Folder)\\ADS - Project 2\\gtfs (1)\\3\\metrotrams\\metro_tram_stops.txt"


In [29]:
# Build a tree of metro tram stop latitude/longitudes
tramstop_coords = np.vstack([metro_tram_stops["stop_lat"], metro_tram_stops["stop_lon"]]).T
tree = cKDTree(tramstop_coords)

# Find the nearest metro train stop to each place_name in postcodes
postcode_coords = np.vstack([postcodes["latitude"], postcodes["longitude"]]).T
tram_distances, tram_indices = tree.query(postcode_coords, k=1)  

# Save results back to postcodes dataframe
postcodes["nearest_metro_tram_stop_distance"] = tram_distances
postcodes["nearest_metro_tram_stop_index"] = tram_indices  

# Having a look at the updated postcodes df
print(postcodes.head())

   postcode      place_name state_name state_code  latitude  longitude  \
0      3000       Melbourne   Victoria        VIC  -37.8140   144.9633   
1      3001       Melbourne   Victoria        VIC  -37.8140   144.9633   
2      3002  East Melbourne   Victoria        VIC  -37.8167   144.9879   
3      3003  West Melbourne   Victoria        VIC  -37.8101   144.9500   
4      3004       Melbourne   Victoria        VIC  -37.8140   144.9633   

   accuracy  nearest_metro_stop_distance  nearest_metro_stop_index  \
0         4                     0.003649                       828   
1         4                     0.003649                       828   
2         4                     0.002773                       774   
3         4                     0.005969                       722   
4         4                     0.003649                       828   

   nearest_metro_stop_lat  nearest_metro_stop_lon  \
0              -37.810535              144.962157   
1              -37.810535   

### Metro Buses

In [ ]:
# Extracting Metropolitan Bus (4) 'STOP' data:
temp_path = "data/metro_bus_stops.txt"
metro_bus_stops = pd.read_csv(temp_path)

In [ ]:
# Build a tree of metro bus stop latitude/longitudes
bus_stop_coords = np.vstack([metro_bus_stops["stop_lat"], metro_bus_stops["stop_lon"]]).T
tree = cKDTree(bus_stop_coords)

# Find the nearest metro train stop to each place_name in postcodes
postcode_coords = np.vstack([postcodes["latitude"], postcodes["longitude"]]).T
bus_distances, bus_indices = tree.query(postcode_coords, k=1)  
 
# Save results back to postcodes dataframe
postcodes["nearest_metro_bus_stop_distance"] = bus_distances
postcodes["nearest_metro_bus_stop_index"] = bus_indices  

# Having a look at the updated postcodes df
print(postcodes.head())

   postcode      place_name state_name state_code  latitude  longitude  \
0      3000       Melbourne   Victoria        VIC  -37.8140   144.9633   
1      3001       Melbourne   Victoria        VIC  -37.8140   144.9633   
2      3002  East Melbourne   Victoria        VIC  -37.8167   144.9879   
3      3003  West Melbourne   Victoria        VIC  -37.8101   144.9500   
4      3004       Melbourne   Victoria        VIC  -37.8140   144.9633   

   accuracy  nearest_metro_stop_distance  nearest_metro_stop_index  \
0         4                     0.003649                       828   
1         4                     0.003649                       828   
2         4                     0.002773                       774   
3         4                     0.005969                       722   
4         4                     0.003649                       828   

   nearest_metro_stop_lat  nearest_metro_stop_lon  \
0              -37.810535              144.962157   
1              -37.810535   

In [ ]:
# Save complete postcodes df to a new csv:
postcodes.to_csv("data/postcodes_with_public_transport_distances.csv", index=False)
       

OSError: Cannot save file into a non-existent directory: '..\data'